# tl;dr
The original Tetra-1.7B corpus is 86.57% low difficulty and only 0.92% hard; the prior recovery add-on is 99.63% critical. The frozen model capped 57/128 recovery games and made parsed illegal terminal actions in the other 71. This notebook reads the immutable audit output and reproduces the core ratios used to design the progressive curriculum.

## Context & Methods
The source audit streams every original row, checks schema, labels, state/prompt identity, split overlap, and coverage, and validates a deterministic 2,000-row teacher sample. Development failures are aggregated without exporting held-out boards or prefixes. Difficulty bands are mutually exclusive and defined in `scripts/audit_tetra17_data.py`.

In [1]:
import json
from pathlib import Path
audit = json.loads(Path('../experiments/tetra17-recovery-v2/audit/data-and-failures.json').read_text())
original = audit['original']
n = original['counts']['rows']
shares = {k: round(v / n, 6) for k, v in original['coverage']['difficulty'].items()}
print('original rows', n)
print('difficulty shares', shares)
print('recovery outcomes', audit['development_failure_profile']['outcomes'])

original rows 1005785
difficulty shares {'critical': 0.035985, 'hard': 0.009155, 'low': 0.865672, 'moderate': 0.089188}
recovery outcomes {'cap_reached': 57, 'illegal_action': 71}


## Data & Results
The original corpus has no missing required fields, invalid rows, prompt-label conflicts, state-label conflicts, or train/eval seed overlap. It has 104,245 serialized prompts shared by multiple board geometries, with no conflicting canonical labels observed. The prior recovery add-on contains only 17 non-critical full-state examples.

In [2]:
assert not original['missing_required_fields']
assert not original['invalid']
assert original['split_overlap']['eval:train'] == {'game_ids': 0, 'seeds': 0}
assert original['identity']['unique_conflicting_prompts'] == 0
assert original['teacher_validation']['teacher_optimal_labels'] == 2000
assert audit['development_failure_profile']['held_out_state_material_emitted'] is False
print('all audit invariants passed')

all audit invariants passed


## Takeaways
The evidence supports a controlled progressive-data test while keeping the teacher, 1.7B base, and prompt representation fixed. It does not establish a causal link between coverage and failure. The preregistered matched evaluation is the test of that hypothesis; conditional RL runs only after every SFT gate passes.